In [1]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import models
from tensorflow.keras import layers
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv("../../dataset/Cleaned_Suicide_Detection_DL.csv")
df.columns

Index(['text', 'class', 'clean_text', 'labels'], dtype='object')

In [3]:
X = df["clean_text"]
y = df["labels"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
tokenizer = Tokenizer(num_words=20000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

In [7]:
model = tf.keras.Sequential([
    layers.Embedding(20000, 128),
    layers.Conv1D(128, 5, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.fit(X_train_pad, y_train, epochs=5, batch_size=32, validation_split=0.1)

Epoch 1/5
5221/5221 ━━━━━━━━━━━━━━━━━━━━ 258s 49ms/step - accuracy: 0.9195 - loss: 0.2131 - val_accuracy: 0.9324 - val_loss: 0.1789
Epoch 2/5
5221/5221 ━━━━━━━━━━━━━━━━━━━━ 252s 48ms/step - accuracy: 0.9484 - loss: 0.1431 - val_accuracy: 0.9358 - val_loss: 0.1766
Epoch 3/5
5221/5221 ━━━━━━━━━━━━━━━━━━━━ 256s 49ms/step - accuracy: 0.9663 - loss: 0.0952 - val_accuracy: 0.9327 - val_loss: 0.1921
Epoch 4/5
5221/5221 ━━━━━━━━━━━━━━━━━━━━ 256s 49ms/step - accuracy: 0.9799 - loss: 0.0563 - val_accuracy: 0.9297 - val_loss: 0.2391
Epoch 5/5
5221/5221 ━━━━━━━━━━━━━━━━━━━━ 251s 48ms/step - accuracy: 0.9870 - loss: 0.0355 - val_accuracy: 0.9288 - val_loss: 0.3015


In [8]:
loss, acc = model.evaluate(X_test_pad, y_test)
print(acc)

1451/1451 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.9275 - loss: 0.3097
0.9275462031364441


In [9]:
model.save("../../models/cnn_model.h5")

In [6]:
model = models.load_model("../../models/bilstm_model.h5")

sample = ["I want to die"]
sample = tokenizer.texts_to_sequences(sample)
sample = pad_sequences(sample, maxlen=100)

pred = model.predict(sample)[0]
pred

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 368ms/step


array([0.92679185], dtype=float32)

In [7]:
y_pred = model.predict(X_test_pad)
y_pred = (y_pred > 0.5).astype(int)
y_pred

1451/1451 ━━━━━━━━━━━━━━━━━━━━ 33s 23ms/step


array([[1],
       [0],
       [1],
       ...,
       [1],
       [1],
       [0]], shape=(46402, 1))

In [8]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.92      0.93     23257
           1       0.92      0.94      0.93     23145

    accuracy                           0.93     46402
   macro avg       0.93      0.93      0.93     46402
weighted avg       0.93      0.93      0.93     46402

